# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a provider tariff
- a distributor tariff
- fees (which are essentially a government tariff)
- a tax rate (a percentage applied to all non government tariff cost)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

In [9]:
import pandas as pd

from energy_cost import Contract, Tariff
from energy_cost.data.be import distributors, fees, tax_rate

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=distributors["fluvius_imewo"],
    fees=fees["flanders_residential"],
    tax_rate=tax_rate,
)

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

contract.calculate_cost(consumption)

timestamp    provider               distributor             \
                            consumption         total consumption              
                                 energy  total  total      energy      total   
0 2024-01-01 00:00:00+01:00       59.52  59.52  59.52   23.676520  23.676520   
1 2024-02-01 00:00:00+01:00       55.68  55.68  55.68   22.149003  22.149003   
2 2024-03-01 00:00:00+01:00        0.02   0.02   0.02    0.007956   0.007956   

                                                      taxes      total  
   capacity            fixed                total     total      total  
      total data_manangement     total      total     total      total  
0  8.209764         1.209508  1.209508  33.095793  5.556948  98.172740  
1  8.209764         1.131475  1.131475  31.490243  5.230215  92.400457  
2  8.209764         0.000406  0.000406   8.218127  0.494288   8.732414

> Note: most tariffs are based on indexes, make sure to define all required indexes before calculating costs. See the [index documentation](./index.ipynb) for more details. If you are definingn your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.

### Resolution

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [10]:
import isodate

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

contract.calculate_cost(consumption, resolution=isodate.Duration(years=1))

timestamp    provider                                \
                            consumption              fixed              
                                 energy   total  fixed_fee      total   
0 2024-01-01 00:00:00+01:00      702.72  702.72    0.00000    0.00000   
1 2025-01-01 00:00:00+01:00      630.72  630.72  120.00000  120.00000   
2 2026-01-01 00:00:00+01:00      135.96  135.96   30.00504   30.00504   

             distributor                                                      \
       total consumption                capacity            fixed              
       total      energy       total       total data_manangement      total   
0  702.72000  279.535692  279.535692   98.517173        14.280000  14.280000   
1  750.72000  412.792925  412.792925  133.105596        17.510000  17.510000   
2  165.96504   59.240491   59.240491   33.875614         2.885852   2.885852   

                   taxes        total  
        total      total        total  
        total      total        total  
0  392.332864  65.703172  1160.756036  
1  563.408521  78.847711  1392.976232  
2   96.001957  15.718020   277.685017

### Time range

By default start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is useful for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [11]:
import datetime as dt
from zoneinfo import ZoneInfo

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003]
})

contract.calculate_cost(consumption, start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")), end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")))

timestamp    provider                                \
                             consumption              fixed              
                                  energy   total  fixed_fee      total   
0  2025-01-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
1  2025-02-01 00:00:00+01:00      725.76  725.76  10.000000  10.000000   
2  2025-03-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
3  2025-04-01 00:00:00+01:00      777.60  777.60  10.000000  10.000000   
4  2025-05-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
5  2025-06-01 00:00:00+01:00      777.60  777.60  10.000000  10.000000   
6  2025-07-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
7  2025-08-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
8  2025-09-01 00:00:00+01:00      777.60  777.60  10.000000  10.000000   
9  2025-10-01 00:00:00+01:00      803.52  803.52  10.000000  10.000000   
10 2025-11-01 00:00:00+01:00      777.60  777.60  10.000000  10.000000   
11 2025-12-01 00:00:00+01:00      777.60  777.60   9.677419   9.677419   

               distributor                                                    \
         total consumption               capacity            fixed             
         total      energy       total      total data_manangement     total   
0   813.520000  525.886877  525.886877  11.092133         1.487151  1.487151   
1   735.760000  474.994598  474.994598  11.092133         1.343233  1.343233   
2   813.520000  525.886877  525.886877  11.092133         1.487151  1.487151   
3   787.600000  508.922784  508.922784  11.092133         1.439178  1.439178   
4   813.520000  525.886877  525.886877  11.092133         1.487151  1.487151   
5   787.600000  508.922784  508.922784  11.461871         1.439178  1.439178   
6   813.520000  525.886877  525.886877  11.831609         1.487151  1.487151   
7   813.520000  525.886877  525.886877  12.201346         1.487151  1.487151   
8   787.600000  508.922784  508.922784  12.571084         1.439178  1.439178   
9   813.520000  525.886877  525.886877  12.940822         1.487151  1.487151   
10  787.600000  508.922784  508.922784  13.310560         1.439178  1.439178   
11  787.277419  508.922784  508.922784  13.310560         1.439178  1.439178   

                    taxes        total  
         total      total        total  
         total      total        total  
0   538.466160  81.119170  1433.105330  
1   487.429964  73.391398  1296.581362  
2   538.466160  81.119170  1433.105330  
3   521.454095  78.543246  1387.597341  
4   538.466160  81.119170  1433.105330  
5   521.823833  78.565430  1387.989263  
6   539.205636  81.163538  1433.889174  
7   539.575374  81.185722  1434.281096  
8   522.933046  78.631983  1389.165029  
9   540.314849  81.230091  1435.064940  
10  523.672522  78.676351  1389.948873  
11  523.672522  78.656996  1389.606937

As you can see in the above example, while the capacity was constant in 2025, the first months are still cheaper then the last, as the cost of the capacity tariff is based on the consumption of the previous 12 months, which includes the cheaper months of 2024.

### Injection

Aside from a consumption timeseries, you can also provide injection as a separate timeseries, as these are often measured independently and have different tariffs. If you provide injection data, the cost of the injection will be calculated separately and subtracted from the cost of the consumption, as injection is essentially negative consumption.

In [14]:
injection = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

consumption = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
    "value": 0.0002
})

contract.calculate_cost(consumption, injection=injection, resolution=isodate.Duration(years=1))

timestamp    provider                                        \
                            consumption         injection               fixed   
                                 energy   total    energy    total  fixed_fee   
0 2024-01-01 00:00:00+01:00      702.72  702.72  -140.544 -140.544    0.00000   
1 2025-01-01 00:00:00+01:00      630.72  630.72  -105.120 -105.120  120.00000   
2 2026-01-01 00:00:00+01:00      135.96  135.96   -28.325  -28.325   30.00504   

                        distributor                                           \
                  total consumption                capacity            fixed   
       total      total      energy       total       total data_manangement   
0    0.00000  562.17600  279.535692  279.535692   98.517173        14.280000   
1  120.00000  645.60000  412.792925  412.792925  133.105596        17.510000   
2   30.00504  137.64004   59.240491   59.240491   33.875614         2.885852   

                              taxes        total  
                   total      total        total  
       total       total      total        total  
0  14.280000  392.332864  57.270532  1011.779396  
1  17.510000  563.408521  72.540511  1281.549032  
2   2.885852   96.001957  14.018520   247.660517